# 02 · Preparar todas las imágenes elegibles para trabajar en local

Aplicamos el alcance y los controles de 01. **No recortamos cada clase hasta alcanzar una cifra.** Preparamos todas las fotos que los superan, conservando sus diferentes frecuencias, y separamos grupos de producto completos.

El conjunto vigente tiene un único `CURRENT.json`. Manifiestos, hashes y ejecuciones internas permiten reproducirlo y auditarlo. Los loaders utilizan las copias preparadas; el RAW sigue conservado hasta verificar una copia externa.

> **Ejecución:** pendiente de volver a ejecutar con la cohorte completa y el protocolo coherente. Los resultados anteriores no se usan como evidencia de esta cohorte.

## Relación con las vías comentadas en clase

Interpretación de los apuntes aproximados de Iratxe (20/11/2026); el mapa completo está al comienzo del notebook 01. La preparación común permite comparar arquitecturas desde cero y modelos preentrenados sin cambiar arbitrariamente los productos de cada partición. Este notebook prepara imágenes: no tokeniza descripciones ni calcula word embeddings.

Para scratch usamos los estadísticos de train. Un backbone preentrenado debe respetar el preprocesamiento asociado a sus pesos, documentando cualquier resolución o normalización diferente. Preentrenado no significa automáticamente autosupervisado: se registrarán el checkpoint, su objetivo de entrenamiento original y sus datos de origen. Cualquier rama NLP deberá heredar las particiones por producto, ajustar vocabulario y transformaciones propias solo con train y publicar su configuración aparte. La versión visual actual no se redefine para añadir esa extensión.


## Protocolo común del proyecto

Primero definimos las prendas que queremos clasificar. Después aplicamos los mismos controles de calidad a todas las fotos y conservamos **todas las que los superan**, aunque unas clases tengan más ejemplos. No se fija un número total ni se igualan las clases con cupos.

Las particiones se construyen sobre todos los grupos de producto elegibles. Los productos ya utilizados durante el desarrollo conservan su exposición: un producto de train no se convierte en un supuesto test nuevo. Los grupos nuevos completan aproximadamente 70/15/15 por clase. Se publican los recuentos reales.

**Antes de recortar train por coste, hay que medirlo.** El baseline usa todos los productos elegibles; 04 registra una época piloto y la estimación de tiempo/memoria. Si fuese necesario reducir datos en otro experimento, se justificaría con esa medición, solo en train y conservando validation/test. Una fotografía fuera de una muestra de cálculo no sería una fotografía de mala calidad.

La referencia es `config/experimental_protocol_v1.json`: macro-F1 primaria, accuracy y resultados por clase; pérdida inicial sin pesos, sin aumentos; normalización solo con train a la resolución de entrada. Tres semillas y coste computacional. Los pesos de clase se estudian como ablación: con frecuencias desiguales ya no suponemos que sean casi uno. El test predictivo se reserva hasta congelar las decisiones. Las métricas describen las doce clases elegidas tras los controles, no todo el catálogo.

## 1. Entorno y contrato de entrada

Se usa la misma configuración que en el notebook 01. Una auditoría completa es obligatoria. Si cambian configuración, auditoría o código, se publica una ejecución nueva; si coinciden, se verifican y reutilizan los resultados. Una interrupción conserva el `CURRENT.json` anterior.

In [ ]:
from pathlib import Path
import sys, json, platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Markdown
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/data/data_contract.py').is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src/data'))
from notebook_utils import setup, styled, es, numeric_axis, save_figure, SPLIT_COLOURS, BLUE, ORANGE
setup()
config = json.loads((ROOT / 'config/local_images_v1.json').read_text(encoding='utf-8'))
SEED = config['seed']
display(Markdown(f"**Proyecto:** `{ROOT.name}` · **Python:** {platform.python_version()} · **Semilla:** {SEED}"))

In [ ]:
from audit_images import read_audit
from prepare_local import prepare, verify_run
from data_contract import sha256
audit_run, audit, audit_summary = read_audit()
display(pd.Series(config, name='Configuración de esta ejecución').to_frame())

## 2. Crear o reutilizar la publicación

El preparador verifica el hash RAW de cada imagen seleccionada antes de transformarla. Aplica orientación EXIF, conversión RGB, ajuste con relleno blanco a 224×224 y compresión JPEG. Repite la detección de igualdad de píxeles después de la transformación, ya que dos originales distintos podrían terminar siendo idénticos.

Se vuelven a conectar los grupos por producto y por hashes de píxeles originales y preparados. Los conflictos de etiqueta se excluyen y los duplicados exactos se reducen a un representante. Después se divide cada clase por grupos con semilla fija. Se calculan los pesos inversos y los momentos de los píxeles exclusivamente sobre entrenamiento.

Los grupos ya utilizados durante el desarrollo conservan su partición según `config/split_exposure.csv`. Los nuevos completan las proporciones por clase. Si una conexión nueva une exposiciones distintas, se conserva la más utilizada: train antes que validation, y validation antes que test. No se convierte una imagen ya vista en un test supuestamente nuevo.

In [ ]:
import contextlib
log_path = ROOT / 'tmp/data_preparation.log'
log_path.parent.mkdir(exist_ok=True)
with log_path.open('w', encoding='utf-8') as log, contextlib.redirect_stdout(log):
    run = prepare(config=config, workers=4)
manifest = verify_run(run, verify_assets=True)
reports = {name: json.loads((run / f'{name}.json').read_text(encoding='utf-8'))
           for name in ['summary', 'labels', 'training_class_weights', 'normalization', 'provenance']}
summary = reports['summary']
usable = manifest.loc[manifest.split.ne('excluded')].copy()
display(Markdown(f"**Versión vigente:** `{run.relative_to(ROOT).as_posix()}`"))
display(styled(pd.DataFrame({'Imágenes': usable.split.value_counts()}).reindex(['train', 'validation', 'test'])))

## 3. Auditoría de exclusiones y contrato de las particiones

Cada fila mantiene el identificador y la razón de exclusión. Una imagen excluida tiene `split=excluded` y no entra en ningún loader. Los pesos de entrenamiento son cero fuera de train. En validación y test no se hacen aumentos ni sobremuestreo.

Las proporciones se aplican al número de grupos de cada clase. Como los grupos tienen distintos tamaños, el reparto de imágenes es aproximado y debe comprobarse con los números publicados.

In [ ]:
exclusions = manifest.loc[manifest.split.eq('excluded'), 'reason'].value_counts().rename_axis('Motivo').to_frame('Artículos')
display(styled(exclusions))
distribution = pd.crosstab(usable.label, usable.split).reindex(columns=['train', 'validation', 'test'])
distribution['Total'] = distribution.sum(axis=1)
display(styled(distribution))
actual = usable.split.value_counts().reindex(['train', 'validation', 'test'])
ratios = pd.DataFrame({'Imágenes': actual, 'Porcentaje real': 100*actual/len(usable),
                      'Porcentaje objetivo por grupos': [100*(1-config['val_size']-config['test_size']), 100*config['val_size'], 100*config['test_size']]})
display(styled(ratios, 2))
assert manifest.loc[manifest.split.eq('excluded'), 'reason'].ne('').all()
assert manifest.loc[manifest.split.ne('train'), 'training_weight'].eq(0).all()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
base = np.zeros(len(distribution))
for split in ['train', 'validation', 'test']:
    values = distribution[split].to_numpy()
    ax.barh(distribution.index, values, left=base, label=split, color=SPLIT_COLOURS[split])
    base += values
ax.set(title='Cada clase mantiene los tres conjuntos separados por producto', xlabel='Imágenes preparadas')
ax.xaxis.grid(True); ax.set_axisbelow(True); numeric_axis(ax)
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.1), ncol=3, frameon=True, facecolor='white', edgecolor='#DDE3EA', fancybox=True)
save_figure(fig, ROOT, '02_class_splits', 'Fuente: manifiesto preparado · Semilla 42 · Grupos conectados de producto y contenido')
plt.show()

## 4. Controles explícitos de fugas y duplicados

No basta con separar identificadores: variantes del mismo producto o copias de una fotografía pueden inflar artificialmente la evaluación. Por eso se comprueban todos los vínculos. Los checksums de la publicación y de todas las imágenes ya se han verificado al cargarla.

La comprobación no garantiza ausencia de fotografías casi idénticas, sesiones fotográficas comunes o errores semánticos de etiquetas; haría falta una auditoría adicional para medirlos.

El test corresponde a grupos de producto no vistos dentro de estas doce clases de H&M. No mide generalización a todo el catálogo, otras marcas o fotografías tomadas en contextos distintos.

In [ ]:
checks = []
for key in ['article_id', 'product_code', 'group_id', 'raw_group_id', 'pixel_sha256', 'prepared_pixel_sha256']:
    crossings = int(usable.groupby(key).split.nunique().gt(1).sum())
    checks.append({'Clave': key, 'Cruces entre particiones': crossings})
    assert crossings == 0
assert usable.prepared_pixel_sha256.is_unique
assert usable.article_id.is_unique
assert set(usable.label) == set(config['include_classes'])
assert not manifest.reason.isin(['class_budget','visual_audit_group']).any()
assert not usable.loc[usable.history_split.eq('train'), 'split'].ne('train').any()
display(styled(pd.DataFrame(checks).set_index('Clave')))
provenance = reports['provenance']
display(pd.Series(provenance['packages'], name='Versión instalada').to_frame())

## 5. Normalización y pesos aprendidos únicamente de entrenamiento

Cada canal se transforma con `(x - media_train) / desviación_train`, después de convertir los píxeles a [0,1]. Los estadísticos son momentos poblacionales ponderados por píxeles de todos los JPEG de train. El relleno forma parte de la imagen preparada y se incluye en estos estadísticos.

Los pesos de clase son `n_train / (n_clases × n_train_clase)`. Se guardan como opción para una ablación de la pérdida; guardarlos no implica que el entrenamiento ya esté equilibrado. Para backbones preentrenados habrá que utilizar el preprocesamiento específico de sus pesos, en lugar de adoptar automáticamente estos estadísticos.

In [ ]:
stats = reports['normalization']
display(styled(pd.DataFrame({'Media [0,1]': stats['mean'], 'Desviación [0,1]': stats['std']}, index=['R', 'G', 'B']), 5))
weights = pd.Series(reports['training_class_weights'], name='Peso de la pérdida').to_frame()
weights['Imágenes de entrenamiento'] = usable.loc[usable.split.eq('train'), 'label'].value_counts()
display(styled(weights, 5))
assert stats['partition'] == 'train'
expected_weights = weights['Imágenes de entrenamiento'].sum() / (len(weights)*weights['Imágenes de entrenamiento'])
np.testing.assert_allclose(weights['Peso de la pérdida'], expected_weights)
assert stats['pixels'] == int(usable.split.eq('train').sum()) * config['image_size']**2

## 6. Dataset y lotes locales sin acceso al RAW

`src/data/local_dataset.py` devuelve imágenes `float32` en formato canales × alto × ancho y etiquetas enteras. Los lotes NumPy permiten verificar esta etapa sin instalar todavía un framework de entrenamiento. El adaptador `make_torch_loaders` usa los mismos archivos y transformaciones cuando se disponga de PyTorch; no se instala ni descarga ningún modelo en esta etapa.

Solo train puede activar aumentos. Validación y test son deterministas. El test no se abre en este notebook: se verifican sus registros y hashes, mientras que las pruebas de carga y la galería usan train/validation.

In [ ]:
from local_dataset import LocalImageDataset, make_torch_loaders
import time, importlib.util
train_plain = LocalImageDataset('train', augment=False, normalize=False, seed=SEED)
train_aug = LocalImageDataset('train', augment=True, normalize=False, seed=SEED)
validation = LocalImageDataset('validation', normalize=True, seed=SEED)
start = time.perf_counter()
x, y = next(validation.batches(batch_size=32))
elapsed = time.perf_counter() - start
assert x.shape == (32, 3, config['image_size'], config['image_size'])
assert x.dtype == np.float32 and y.dtype == np.int64
assert np.isfinite(x).all() and y.min() >= 0 and y.max() < len(reports['labels'])
a, b = validation[0], validation[0]
np.testing.assert_array_equal(a[0], b[0])
display(styled(pd.DataFrame({'Valor': [len(train_plain), len(validation), elapsed, len(y)/elapsed]},
                           index=['Imágenes train', 'Imágenes validation', 'Segundos primer lote (incluye apertura)', 'Imágenes/s primer lote']), 3))
try:
    LocalImageDataset('test', augment=True)
    raise AssertionError('Se permitieron aumentos de test')
except ValueError:
    print('Control correcto: aumentos prohibidos fuera de train.')
if importlib.util.find_spec('torch'):
    import torch
    loaders = make_torch_loaders(batch_size=8, seed=SEED)
    tx, ty = next(iter(loaders['train']))
    assert tuple(tx.shape) == (8, 3, config['image_size'], config['image_size'])
    print('PyTorch:', torch.__version__, '· CUDA disponible:', torch.cuda.is_available())
else:
    print('PyTorch no está instalado: lotes NumPy comprobados; adaptador PyTorch pendiente de validar con ese framework.')

## 7. Comprobación visual de los aumentos de train

Se permiten inversión horizontal y variaciones leves de brillo y color. No se aplican giros grandes ni recortes que eliminen partes de la prenda. La galería es una comprobación inicial, insuficiente para garantizar por sí sola que todos los aumentos conserven la información de clase. El efecto real de estos aumentos se estudiará mediante una ablación con la misma partición.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
for j in range(4):
    original, label = train_plain[j]
    augmented, same_label = train_aug[j]
    assert label == same_label
    axes[0, j].imshow(original.transpose(1, 2, 0))
    axes[1, j].imshow(augmented.transpose(1, 2, 0))
    name = train_plain.rows.iloc[j].label
    axes[0, j].set_title(f'Original preparado · {name}', fontsize=9, weight='normal')
    axes[1, j].set_title('Aumento de entrenamiento', fontsize=9, weight='normal')
for ax in axes.flat: ax.axis('off')
fig.suptitle('Los aumentos leves conservan la forma de la prenda', fontsize=15, weight='bold', y=.98)
save_figure(fig, ROOT, '02_training_augmentation', 'Fuente: cuatro imágenes de train H&M · Semilla 42 · No se muestran imágenes de test')
plt.show()

## 8. Espacio local y una sola publicación vigente

Las fotos se guardan en `data/processed/images/<prefijo>/<sha256>.jpg`, con rutas relativas. `CURRENT.json` señala el manifiesto completo vigente. Los archivos con los mismos bytes se reutilizan; las publicaciones internas anteriores son trazabilidad, no una segunda alternativa del proyecto.

Comparamos el espacio de las fotos elegibles originales con el de sus mismas copias preparadas. El ahorro por redimensionado no debe confundirse con elegir menos imágenes. El ZIP y los originales siguen locales hasta comprobar su traslado externo; todavía no contamos ese espacio como liberado.

In [ ]:
same_original_bytes = usable.file_bytes.sum()
size_table = pd.DataFrame({'MB decimales': [same_original_bytes/1e6, summary['prepared_bytes']/1e6],
                           'Fotografías': [len(usable), len(usable)]},
                          index=['Originales de las fotos elegibles', 'Las mismas fotos preparadas'])
display(styled(size_table, 2))
reduction = 100*(1-summary['prepared_bytes']/same_original_bytes)
display(Markdown(f"Las {es(len(usable))} fotos preparadas ocupan **{es(summary['prepared_bytes']/1e6, 1)} MB**. Frente a sus mismos originales, el ahorro es **{es(reduction, 2)} %**. Esta comparación no atribuye al redimensionado el ahorro por excluir categorías."))
pointer = json.loads((ROOT / 'data/processed/CURRENT.json').read_text())
assert (ROOT / 'data/processed' / pointer['run_dir']) == run

## 9. Qué tenemos y cómo seguimos

Queda una cohorte local completa después del alcance y la calidad, con todas sus exclusiones explicadas, etiquetas, grupos, particiones y estadísticos. Las clases no tienen por qué sumar lo mismo. 03 realiza EDA y baselines sobre esta publicación; 04 mide el coste y entrena la Dense con sus tres semillas y la ablación de dropout.

La pérdida inicial no lleva pesos de clase. El desequilibrio se estudia con macro-F1 y resultados por clase, y las estrategias de ponderación se reservan para ablaciones. Los originales siguen preservados; las copias de RGB224 permiten derivar entradas menores y no recuperan el detalle original.

In [ ]:
display(Markdown(f"**Publicación verificada:** {es(len(usable))} imágenes, {es(usable.label.nunique())} clases. "
                 f"Train: {es(usable.split.eq('train').sum())}; "
                 f"validation: {es(usable.split.eq('validation').sum())}; "
                 f"test: {es(usable.split.eq('test').sum())}. "
                 "Sin cruces de producto ni de píxeles exactos entre particiones."))